# 使用看似不相关回归估计住宅能源需求系统

## 执行摘要

一家区域性公用事业公司使用 **PROC SYSLIN** 的看似不相关回归（SUR）方法，估计一个两方程的住宅**能源需求系统**——电力与天然气的预算份额作为自价格、交叉价格和家庭收入的函数。这两个份额方程共享一个共同的家庭预算，因此它们的误差是交叉相关的；SUR 联合估计这些方程以利用该相关性，恢复出符号一致的自价格与交叉价格效应，并提供需求分析师所需的跨方程协方差与约束检验。

## 数据来源

| 数据集 | 行数 | 粒度 | 关键变量 | 说明 |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | 每月市场观测一行 | `month`、`lp_elec`、`lp_gas`、`lincome`、`w_elec`、`w_gas` | 合成的月度住宅能源市场面板。`lp_elec`/`lp_gas` 是电力与天然气的对数实际价格；`lincome` 是对数实际家庭收入；`w_elec`/`w_gas` 是电力与天然气的支出预算份额，由一个已知的 AIDS 式需求结构加上相关的跨方程噪声生成。 |

# 使用看似不相关回归估计住宅能源需求系统

一家区域性燃气电力公用事业公司希望了解家庭如何随相对价格和收入的变化，在**电力**与**天然气**之间重新分配其能源预算。自然的框架是一个**需求系统**：一组联合估计的预算份额方程。

有两个特征使联合估计成为正确的工具：

- 份额方程取自共同的家庭预算，因此它们的误差是**交叉相关的**——当一个家庭在电力上花费更多时，它在天然气上花费更少。用**看似不相关回归（SUR）**将方程一起估计，可利用该相关性提高效率。
- 经济理论施加了**线性约束**（加总性、齐次性、对称性），系统估计量可以直接强制执行这些约束。

`PROC SYSLIN` 配合 `SUR` 选项正是为这种情形而设计。它拟合每个份额方程，估计跨方程误差协方差，并用该协方差重新拟合系统，从而提供高效、理论一致的弹性——连同跨模型协方差矩阵和联合约束检验。

在本 notebook 中我们：

1. 从一个已知的份额结构生成合成的月度能源市场面板。
2. 用 SUR 拟合一个两方程份额系统。
3. 将联合 SUR 拟合与逐方程 OLS 进行比较。
4. 施加齐次性约束并读取其联合 F 检验。
5. 提取系数表用于弹性报告。

## 步骤 1 — 生成合成能源市场面板

我们模拟一个小型**双商品能源需求系统**（电力与天然气）的 60 个月度观测。数据生成过程是一个线性化的 AIDS 式预算份额模型：

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

我们有意构建**跨方程相关性**：当家庭在电力上花费更多时，它们在天然气上花费更少，因此 `e1` 和 `e2` 是负相关的。一个共同的能源市场价格因子也驱动两个对数价格一起变动。正是这些特征使联合 SUR 估计优于单独拟合每个方程。

In [1]:
数据 energy;
   调用 streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   循环 month = 1 到 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      输出;
   结束;

   保留 month lp_elec lp_gas lincome w_elec w_gas;
运行;

过程 均值 数据=energy n mean std MIN MAX maxdec=3;
   变量 w_elec w_gas lp_elec lp_gas lincome;
   标签 w_elec="电力预算份额" w_gas="天然气预算份额" lp_elec="对数电价(LP_ELEC)" lp_gas="对数气价(LP_GAS)" lincome="对数收入(LINCOME)";
运行;

                                                  The MEANS Procedure

 Variable  Label                        N           Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 w_elec    电力预算份额                      60          0.440       0.011       0.417       0.464
 w_gas     天然气预算份额                     60          0.228       0.014       0.198       0.256
 lp_elec   对数电价(LP_ELEC)               60          4.617       0.142       4.354       4.911
 lp_gas    对数气价(LP_GAS)                60          4.211       0.162       3.818       4.566
 lincome   对数收入(LINCOME)               60         10.916       0.088      10.723      11.174
 -------------------------------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 步骤 2 — 需求系统的基线 SUR 估计

我们联合估计两个份额方程。`SUR` 选项告诉 `PROC SYSLIN` 估计跨方程误差协方差，并用它进行高效的可行 GLS 拟合。两个带标签的 `MODEL` 语句（`elec` 和 `gas`）定义了系统；每个都将一个预算份额对两个对数价格和对数收入回归。

SYSLIN 报告每个方程的参数估计，并在最后报告**跨模型协方差矩阵**——SUR 所利用的两个方程误差的估计协方差。

In [2]:
过程 syslin 数据=energy sur;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
   标签 w_elec="电力预算份额" w_gas="天然气预算份额" lp_elec="对数电价" lp_gas="对数气价" lincome="对数收入";
运行;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 电力预算份额

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  对数电价           1      0.112463    0.021244       5.29      0.0000
  对数气价           1     -0.097938    0.018646      -5.25      0.0000
  对数收入           1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: 天然气预算份额

  Number of Observations                       60
  SSE                                      


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## 步骤 3 — 与逐方程 OLS 比较

为了看清 SUR 给我们带来了什么，我们用普通最小二乘法（默认方法，一次一个方程）重新拟合相同的两个方程。OLS 忽略跨方程误差相关性，因此当该相关性非零时（此处按构造即如此），它是一致的但不如 SUR 高效。

比较两张系数表可以看到，一旦考虑系统结构，估计值及其标准误如何变化。

In [3]:
过程 syslin 数据=energy ols;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
   标签 w_elec="电力预算份额" w_gas="天然气预算份额" lp_elec="对数电价" lp_gas="对数气价" lincome="对数收入";
运行;



                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: 电力预算份额

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  对数电价           1      0.112463    0.021989       5.11      0.0000
  对数气价           1     -0.097938    0.019300      -5.07      0.0000
  对数收入           1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: 天然气预算份额

  Number of Observations                       60
  SSE                                      


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## 步骤 4 — 施加经济理论并检验它

需求理论认为名义价格/收入效应应服从**零次齐次性**：对所有价格和收入按比例缩放，实际需求不变，因此一个方程内的价格和收入系数之和为零。我们用 `RESTRICT` 语句将其施加于电力份额上。

SYSLIN 在该约束下重新拟合系统，并报告一个**约束结果**F 检验，检验该约束是否与数据一致——即对齐次性假设的直接联合检验。

In [4]:
过程 syslin 数据=energy sur;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
   标签 w_elec="电力预算份额" w_gas="天然气预算份额" lp_elec="对数电价" lp_gas="对数气价" lincome="对数收入";
   /* 电力份额方程的齐次性：价格与收入系数之和为零。 */
   restrict lp_elec + lp_gas + lincome = 0;
运行;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 电力预算份额

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  对数电价           1      0.112463    0.021244       5.29      0.0000
  对数气价           1     -0.097938    0.018646      -5.25      0.0000
  对数收入           1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: 天然气预算份额

  Number of Observations                       60
  SSE                                      


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## 步骤 5 — 捕获估计值用于弹性报告

为了最终报告，我们用 `OUTEST=` 持久化收敛后的系数。得到的数据集每个方程一行，包含截距和斜率估计以及拟合统计量，这些为公用事业公司在样本均值份额处的自价格与交叉价格弹性计算提供输入。我们将其打印存档。

In [5]:
过程 syslin 数据=energy sur outest=demand_est;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
   标签 w_elec="电力预算份额" w_gas="天然气预算份额" lp_elec="对数电价" lp_gas="对数气价" lincome="对数收入";
运行;

过程 打印 数据=demand_est noobs;
   标题 "SUR 需求系统系数估计";
运行;
标题;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 电力预算份额

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  对数电价           1      0.112463    0.021244       5.29      0.0000
  对数气价           1     -0.097938    0.018646      -5.25      0.0000
  对数收入           1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: 天然气预算份额

  Number of Observations                       60
  SSE                                      


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## 结果解读

**符号一致性。** SUR 估计恢复了内建于数据中的替代结构。在电力份额方程中，**自对数价格系数为正**（`LP_ELEC` = 0.112），而**交叉价格系数为负**（`LP_GAS` = -0.098）；在天然气份额方程中，该模式与之镜像对称（自 `LP_GAS` = 0.114，交叉 `LP_ELEC` = -0.075）。每一个自价格与交叉价格效应都具有统计显著性（每个 `Pr > |t|` 都低于 0.005），因此替代符号是被牢固识别的，而非噪声拟合的产物。

**SUR 利用跨方程相关性。** **跨模型协方差矩阵**显示一个负的非对角元（-0.000068）：家庭在电力支出与天然气支出之间进行权衡，与构造完全一致。由于该协方差非零，联合 SUR 估计比单独拟合每个方程更高效——步骤 2 中的 SUR 标准误一致地略小于步骤 3 中逐方程 OLS 的标准误（例如，天然气 `LP_ELEC` 的标准误从 OLS 下的 0.0264 降至 SUR 下的 0.0255），而点估计不变。更紧的推断正是对系统而非两个单独回归建模的回报。

**理论成立。** 通过 `RESTRICT` 将**零次齐次性**施加于电力份额（价格与收入系数之和为零）产生了一个约束结果 F 检验值 2.14，`Pr > F` = 0.149。该约束**未被拒绝**，因此消费者需求理论的齐次性预测与本样本一致——公用事业公司可以放心地将受约束的、理论一致的估计带入报告。

**运营用途。** `OUTEST=` 表持久化每个方程一行，含截距和斜率估计以及拟合统计量。这些系数直接转换为样本均值份额处（步骤 1 得出 `W_ELEC` ~ 0.44、`W_GAS` ~ 0.23）的自价格与交叉价格弹性以及收入（支出）弹性，为公用事业公司的需求预测和费率设计情景提供了一个站得住脚、理论一致的依据。